In [129]:
from data.constants import DNC_START, DNC_END
import pandas as pd
import numpy as np

In [130]:
train_panel_in = "../../../data/final/train_panel.parquet"
bike_panel_in = "../../../data/final/bike_panel.parquet"
uber_panel_in = "../../../data/final/uber_panel.parquet"

bal_out = "../../../reports/replication/balance.html"

In [131]:
train_panel = pd.read_parquet(train_panel_in)
bike_panel = pd.read_parquet(bike_panel_in)
uber_panel = pd.read_parquet(uber_panel_in)

In [132]:
panel = pd.concat([train_panel, bike_panel, uber_panel],
                     keys=['train', 'bike', 'uber'], names=['transit']).reset_index('transit')
assert not any(panel.groupby('id')['transit'].nunique() > 1), 'ids must be unique across transits'

In [140]:
treat_group = 'UCMP'

In [141]:

rows = []
for (t,g), sub in panel.groupby(['transit', treat_group]):
    pre = sub[(sub['date'] < DNC_START) | (sub['date'] > DNC_END)]
    post = sub[(sub['date'] >= DNC_START) & (sub['date'] <= DNC_END)]
    rows.append({
        "Group": "Treated" if g == 1 else "Control",
        "Mode": t,
        "N units": sub['id'].nunique(),
        "N unit-days": len(sub),
        "N pre": len(pre),
        "N post": len(post),
        "Mean log(rides) (pre)": pre['logrides'].mean(),
        "Mean log(rides) (post)": post['logrides'].mean(),
        "% complete panel": (sub.groupby('id').size()
                             .eq(panel['date'].nunique()).mean()),
    })
bal = pd.DataFrame(rows).T
bal = bal.map(lambda v: f"{v:.3f}" if isinstance(v, float) else v)
from tabulate import tabulate
header_rows = list(map(lambda x: f'{x[0]}\n{x[1]}', zip(['bike', ' ', 'train', ' ', 'uber', ' '], bal.iloc[0])))
bal_table = tabulate(bal.iloc[2:], headers=header_rows, tablefmt="github", showindex=True)
print(bal_table)

# with open(bal_out,"w") as f:
#     f.write(bal_table)

# !cp {bal_out} ../../../../eric-mc2-cv/static/uploads/

|                        |      bike |           |     train |           |      uber |           |
|                        |   Control |   Treated |   Control |   Treated |   Control |   Treated |
|------------------------|-----------|-----------|-----------|-----------|-----------|-----------|
| N units                |  1549     |    47     |   134     |     8     |  1217     |    40     |
| N unit-days            | 76244     |  4552     | 13132     |   714     | 95398     |  3899     |
| N pre                  | 73081     |  4380     | 12596     |   682     | 91557     |  3739     |
| N post                 |  3163     |   172     |   536     |    32     |  3841     |   160     |
| Mean log(rides) (pre)  |     2.598 |     3.582 |     7.306 |     7.039 |     3.806 |     5.552 |
| Mean log(rides) (post) |     2.579 |     3.887 |     7.35  |     7.386 |     3.679 |     5.563 |
| % complete panel       |     0.274 |     0.809 |     1     |     0.875 |     0.552 |     0.9   |
